In [ ]:
import os
import pandas as pd
from pathlib import Path

ieeg_dir = Path('/users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/ieeg_shortWOBS_fs250')
ieeg_subjects = [p.name.removesuffix('_epochs.p') for p in ieeg_dir.glob('*_epochs.p')]

ieeg_dem = pd.read_csv('/users/barbara/Desktop/MINDLAB2021_MEG-TempSeqAges/scratch/learning_bach_iEEG/misc/clean_dataset.csv')
meg_dem=pd.read_excel('/users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/MEG/SubjectsDataCollection_APR2020_TEMP.xlsx', 'Background questionnaire')

ieeg_dem_=ieeg_dem[['subject', 'age', 'sex', 'instrument']]
ieeg_dem_ = ieeg_dem_[ieeg_dem_['subject'].isin(ieeg_subjects)].reset_index(drop=True)
ieeg_dem_['instrument'] = ieeg_dem_['instrument'].fillna(False).apply(lambda x : 'Yes' if x != False else 'No')
ieeg_dem_=ieeg_dem_.dropna().reset_index(drop=True) # Remove DA037, LL13 (???)
meg_dem_ = meg_dem[['Subject', 'Age', 'Sex', '12. Are you a musician?']].dropna().reset_index(drop=True)
meg_dem_  =meg_dem_.rename(columns={'Subject' : 'subject', 'Age':'age', 'Sex':'sex', '12. Are you a musician?':'instrument'})


In [ ]:
len(meg_dem_)

In [ ]:
len(ieeg_dem_)

## Nearest-neighbor demographic matching

Greedy one-to-one matching without replacement: visit iEEG participants in a reproducibly shuffled order and select the nearest unused MEG participant. The original loading and iEEG instrument Yes/No conversion above are preserved, including the classification of missing instrument entries as No.

Age is standardized using the combined eligible cohorts' sample standard deviation $s_a$. Sex and instrument status are nominal categories. The mixed distance is
$$d(i,j)=\sqrt{w_a[(a_i-a_j)/s_a]^2+w_s\mathbf{1}(s_i\ne s_j)+w_m\mathbf{1}(m_i\ne m_j)}.$$
All three weights default to 1: one categorical mismatch contributes as much squared distance as an age difference of one standard deviation. Sex and instrument matches are preferred through penalties, but are **not required to be exact**. Review mismatch flags and age gaps. Weights encode substantive choices and should be specified before downstream outcome analysis.

Greedy matching depends on visit order and does not minimize total distance globally. The seed fixes the order for reproducibility; ties use MEG subject-ID order. Matching addresses observed demographic similarity only. Instrument playing and self-reported musician status still require substantive comparability.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_STATE = 42
MATCH_WEIGHTS = {'age': 1.0, 'sex': 1.0, 'instrument': 1.0}
EXPORT_RESULTS = False
OUTPUT_DIR = Path('out/demographic_matching')

# Normalize category formatting on copies; preserve your prepared dataframes.
# Add verified aliases here if the source files use other labels.
SEX_ALIASES = {'f': 'female', 'woman': 'female', 'm': 'male', 'man': 'male'}

def prepare_for_matching(frame):
    out = frame[['subject', 'age', 'sex', 'instrument']].copy()
    out['subject'] = out['subject'].astype('string').str.strip()
    if out['subject'].isna().any() or out['subject'].eq('').any() or out['subject'].duplicated().any():
        raise ValueError('Subject IDs must be nonmissing and unique within each cohort.')
    out['age'] = pd.to_numeric(out['age'], errors='raise')
    if not np.isfinite(out['age'].to_numpy(float)).all() or (out['age'] <= 0).any():
        raise ValueError('Ages must be finite and positive.')
    for col in ['sex', 'instrument']:
        out[col] = out[col].astype('string').str.strip().str.lower()
        if out[col].isna().any() or out[col].eq('').any():
            raise ValueError(f'Missing or empty {col} labels.')
    out['sex'] = out['sex'].replace(SEX_ALIASES)
    if not out['instrument'].isin(['yes', 'no']).all():
        raise ValueError('Instrument must contain Yes/No labels in both cohorts.')
    out['instrument'] = out['instrument'].map({'yes': 'Yes', 'no': 'No'})
    return out.sort_values('subject').reset_index(drop=True)

ieeg_eligible = prepare_for_matching(ieeg_dem_)
meg_eligible = prepare_for_matching(meg_dem_)
print(f'Eligible participants: {len(ieeg_eligible)} iEEG; {len(meg_eligible)} MEG')
for name, frame in [('iEEG', ieeg_eligible), ('MEG', meg_eligible)]:
    print(name, 'sex labels:', sorted(frame['sex'].unique()))
if set(ieeg_eligible['sex']) != set(meg_eligible['sex']):
    print('Sex categories differ between cohorts. Verify source coding before interpreting matches.')


In [ ]:
def nearest_neighbor_match(ieeg, meg, weights=None, random_state=42):
    """Greedy nearest available neighbor on standardized age and category mismatches."""
    weights = {'age': 1.0, 'sex': 1.0, 'instrument': 1.0} if weights is None else weights
    if set(weights) != {'age', 'sex', 'instrument'}:
        raise ValueError('Provide weights for age, sex, and instrument.')
    w = np.asarray([weights[k] for k in ['age', 'sex', 'instrument']], dtype=float)
    if not np.isfinite(w).all() or (w < 0).any() or not (w > 0).any():
        raise ValueError('Weights must be finite, nonnegative, and not all zero.')
    if len(ieeg) == 0 or len(meg) < len(ieeg):
        raise ValueError('Require a nonempty iEEG cohort and at least as many MEG participants.')
    age_scale = pd.concat([ieeg['age'], meg['age']], ignore_index=True).std(ddof=1)
    if not np.isfinite(age_scale):
        raise ValueError('Cannot calculate age standard deviation.')
    # When all ages are identical their contribution is zero.
    age_scale = float(age_scale) if age_scale > 0 else 1.0
    age_gap = np.abs(ieeg['age'].to_numpy(float)[:, None] - meg['age'].to_numpy(float)[None, :])
    squared = w[0] * (age_gap / age_scale)**2
    for weight, col in zip(w[1:], ['sex', 'instrument']):
        squared += weight * (ieeg[col].to_numpy()[:, None] != meg[col].to_numpy()[None, :])
    distances = np.sqrt(squared)
    visit_order = np.random.default_rng(random_state).permutation(len(ieeg))
    available = np.ones(len(meg), dtype=bool)
    selected = np.empty(len(ieeg), dtype=int)
    for i in visit_order:
        j = int(np.argmin(np.where(available, distances[i], np.inf)))
        selected[i] = j
        available[j] = False
    # Return pairs in input iEEG order, with the greedy visit order recorded.
    pairs = pd.concat([ieeg.reset_index(drop=True).add_prefix('ieeg_'),
                       meg.iloc[selected].reset_index(drop=True).add_prefix('meg_')], axis=1)
    visit_rank = np.empty(len(ieeg), dtype=int)
    visit_rank[visit_order] = np.arange(1, len(ieeg) + 1)
    pairs['matching_order'] = visit_rank
    pairs['distance'] = distances[np.arange(len(ieeg)), selected]
    pairs['age_gap_years'] = age_gap[np.arange(len(ieeg)), selected]
    pairs['meg_minus_ieeg_age'] = pairs['meg_age'] - pairs['ieeg_age']
    for col in ['sex', 'instrument']:
        pairs[f'{col}_match'] = pairs[f'ieeg_{col}'] == pairs[f'meg_{col}']
    assert pairs['meg_subject'].is_unique
    return pairs, age_scale

matched_pairs, age_scale = nearest_neighbor_match(
    ieeg_eligible, meg_eligible, weights=MATCH_WEIGHTS, random_state=RANDOM_STATE)
matched_meg_subjects = matched_pairs['meg_subject'].tolist()
meg_matched = meg_eligible.set_index('subject').loc[matched_meg_subjects].reset_index()
print(f'Matched {len(matched_pairs)} distinct pairs; age scaling SD = {age_scale:.2f} years')
display(matched_pairs)
display(matched_pairs[['distance', 'age_gap_years', 'meg_minus_ieeg_age']].describe())
print('Categorical mismatches:')
display((~matched_pairs[['sex_match', 'instrument_match']]).sum())


## Balance and pair quality

Signed standardized mean differences (SMDs) compare MEG minus iEEG. The denominator is the pooled **unmatched** standard deviation and is held fixed after matching. Binary covariates use Bernoulli variances; multicategory sex is evaluated with one indicator per category. An absolute SMD of 0.1 is a descriptive reference, not a guarantee of adequate balance. Review raw age distributions and the largest pairwise gaps as well. Greedy nearest-neighbor matching does not guarantee categorical or age balance. Inspect both the cohort-level diagnostics and individual pair mismatches.


In [ ]:
def balance_table(ieeg, meg, selected):
    features = {'Age (years)': (ieeg['age'], meg['age'], selected['age'])}
    for column in ['sex', 'instrument']:
        for level in sorted(set(ieeg[column]) | set(meg[column])):
            features[f'{column}: {level}'] = tuple((frame[column] == level).astype(float)
                                                    for frame in (ieeg, meg, selected))
    records = []
    for name, (target, before, after) in features.items():
        binary = name != 'Age (years)'
        var = lambda x: x.mean() * (1 - x.mean()) if binary else x.var(ddof=1)
        scale = np.sqrt((var(target) + var(before)) / 2)
        def smd(value):
            delta = value.mean() - target.mean()
            return delta / scale if scale > 0 else (0.0 if delta == 0 else np.nan)
        records.append({'covariate': name, 'iEEG_mean_or_proportion': target.mean(),
                        'MEG_before': before.mean(), 'MEG_matched': after.mean(),
                        'SMD_before': smd(before), 'SMD_matched': smd(after)})
    return pd.DataFrame(records).set_index('covariate')

balance = balance_table(ieeg_eligible, meg_eligible, meg_matched)
display(balance.round(3))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
y = np.arange(len(balance))
for col, label in [('SMD_before', 'Before'), ('SMD_matched', 'Matched')]:
    axes[0].scatter(balance[col].abs(), y, label=label)
axes[0].axvline(.1, color='gray', linestyle=':')
axes[0].set(yticks=y, yticklabels=balance.index, xlabel='Absolute SMD (fixed denominator)')
axes[0].legend()
for frame, label in [(ieeg_eligible, 'iEEG'), (meg_eligible, 'MEG before'), (meg_matched, 'MEG matched')]:
    ages = np.sort(frame['age'].to_numpy())
    axes[1].step(ages, np.arange(1, len(ages)+1)/len(ages), where='post', label=label)
axes[1].set(xlabel='Age (years)', ylabel='Empirical cumulative proportion')
axes[1].legend()
axes[2].scatter(matched_pairs['ieeg_age'], matched_pairs['meg_age'])
limits = [min(ieeg_eligible.age.min(), meg_matched.age.min()),
          max(ieeg_eligible.age.max(), meg_matched.age.max())]
axes[2].plot(limits, limits, 'k:', linewidth=1)
axes[2].set(xlabel='iEEG age (years)', ylabel='Matched MEG age (years)')
fig.tight_layout()
plt.show()
print('Largest age gaps:')
display(matched_pairs.sort_values('age_gap_years', ascending=False).head(10))


In [ ]:
# Optional reproducible outputs; matching code remains entirely in this notebook.
if EXPORT_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    matched_pairs.to_csv(OUTPUT_DIR / 'matched_pairs.csv', index=False)
    balance.to_csv(OUTPUT_DIR / 'balance.csv')
    meg_matched.to_csv(OUTPUT_DIR / 'selected_meg_demographics.csv', index=False)
    pd.Series(matched_meg_subjects, name='subject').to_csv(OUTPUT_DIR / 'selected_meg_subjects.csv', index=False)
    print(f'Saved results to {OUTPUT_DIR.resolve()}')
